In [1]:
import numpy as np 
import pandas as pd 
import random 

In [2]:
#goals 
# 1. create '2000-30' synthetic data - >integer
# 2. create a random number generator from 1 to 2000
# 3. sample 30 different values 
# 4. From the original dataset copy data whose index matches the numbers gotten from random number generator
# 5. Reinsert those data into the dataset and hence finishing the goal of number with duplicates

# for dates -> dates 
# sign up dates in future
# last order date, 8%~ missing

# total orders -> integer -> 0.1
# some zeros, despite last_order_date being populated

# average order value -> normal 
# few extreme outliers £0.01, £9999 -> rate = 0.05

# email open rate -> beta
# values slightly above 1.0 (bad tracking data)

# customer_segment -> choice
# Mixed case "VIP", "vip", "Vip"

# subscription status -> choice
# "active", "Active", "ACTIVE", NaN

# age - log-normal distribution 
# 5% missing, a few implausible values (age 7, age 140) [1,11] and [100,151]

# region - choice
# some valid, some "N/A", some empty strings

# churned
# target - defined as no order in 60 days (derived manually)


# strategy
# 

In [32]:
class generate:
    def __init__(self, n, seed=None):
        """constuctor

        Args:
            n (int): number of rows or records in the data to generate.
            seed (int, optional): the seed number for NumPy's default_rng. Defaults to None.
        """        
        self.n = n
        self.rng = np.random.default_rng(seed=seed)

    def customer_id(self):
        """Method to generate customer_id

        Returns:
            ndarray: numpy array of customer_ids
        """        
        ids =  np.arange(1, self.n+1)
        return np.array([f"CUST-{x:04d}" for x in ids])
    
    def dates(self, start_date, end_date, distribution="uniform"):
        """Method to generate different dates

        Args:
            start_date (str): start date of date values of records
            end_date (str): the last date of date value of records
            distribution (str, optional): Uniform or Triangular for recent dates having more impact. Defaults to "uniform".

        Returns:
            ndarray: array of date datetime values
        """        
        start = np.array(start_date, dtype='datetime64[D]')
        end = np.array(end_date, dtype='datetime64[D]')

        total_days = (end-start).astype(int)

        if distribution == "triangular":
            offsets = self.rng.triangular(left=0, mode = total_days, right=total_days, size=self.n)
        else:
            offsets = self.rng.integers(0, total_days, size=self.n, endpoint=True)

        result_dates = start + np.timedelta64(1, 'D') * offsets.astype(int)

        return result_dates
    
    def total_orders(self, lam=3, low=1, high=50):
        """Method to generate total_orders

        Args:
            lam (int, optional): lambda for poisson distribution. Defaults to 3.
            low (int, optional): low value for clipping. Defaults to 1.
            high (int, optional): high value for clipping. Defaults to 50.

        Returns:
            ndarray: clipped numpy array containing total orders per customer.
        """        
        raw_data = self.rng.poisson(lam=lam, size=self.n) + low 
        clipped_data = np.clip(raw_data, low, high)
        return clipped_data
    
    def avg_order_value(self, mean, std, clip = None):
        """Method to generate average order value

        Args:
            mean (int): mean for the normal distribution.
            std (int): standard deviation for the normal distribution.
            clip (list, optional): list containing low and high for clipping. Defaults to None.

        Returns:
            ndarray: clipped numpy array containing average order value per customer
        """        
        data = self.rng.normal(loc = mean, scale=std, size=self.n)
        if clip and len(clip) == 2:
            data = np.clip(data, clip[0], clip[1])
        
        return np.round(data, 2)
    
    def email_open_rate(self, alpha, beta):
        """Method for generating email open rate of customers

        Args:
            alpha (int): alpha value for beta distribution.
            beta (int): beta value for beta distribution.

        Returns:
            ndarray: numpy array containing email open rates.
        """        
        return self.rng.beta(a=alpha, b=beta, size=self.n)
    
    def customer_segment(self, vals=["New", "new", "NEW", "vip", "VIP", "Vip", "loyal", "LOYAL", "Loyal", "at-risk", "At-Risk"], probs = [0.14, 0.13, 0.13, 0.04, 0.07, 0.04, 0.11, 0.12, 0.12, 0.03, 0.07]):
        """Method for generating customer segment

        Args:
            vals (list, optional): different values the column can be. Defaults to ["New", "new", "NEW", "vip", "VIP", "Vip", "loyal", "LOYAL", "Loyal", "at-risk", "At-Risk"].
            probs (list, optional): different probability of each value the column could take. Defaults to [0.14, 0.13, 0.13, 0.04, 0.07, 0.04, 0.11, 0.12, 0.12, 0.03, 0.07].

        Returns:
            ndarray: numpy array containing customer segment.
        """        
        return self.rng.choice(vals, size=self.n, p = probs, replace=True) 

    def subscription_status(self, vals = None, counts = None):
        """Method for generating subscription status

        Args:
            vals (list, optional): list containing different values the column could be. Defaults to None.
            counts (list, optional): list containing the probability of different values the could could be. Defaults to None.

        Returns:
            ndarray: numpy array of subscription statuses of customer.
        """        
        # Safe handling of default list arguments in Python
        if vals is None:
            vals = ["Active", "ACTIVE", "active", "Cancelled", "cancelled", np.nan]
        if counts is None:
            counts = [1100, 100, 263, 400, 87, 20]
        total_data = np.repeat(vals, counts)
        self.rng.shuffle(total_data)
        return total_data 
    
    def age(self, mean= None, sigma = None):
        """Method for generating Age

        Args:
            mean (int, optional): mean of lognormal distribution. Defaults to None.
            sigma (int, optional): standard deviation of the lognormal distribution. Defaults to None.

        Returns:
            ndarray: numpy array of ages of customers.
        """        
        if mean == None:
            mean = 3.5
        if sigma == None:
            sigma = 0.25
        realistic_ages = self.rng.lognormal(mean=mean, sigma=sigma, size=self.n)
        realistic_ages_clipped = np.clip(realistic_ages, 18, 70)
        return np.round(realistic_ages_clipped).astype(int)
    
    def region(self, vals = None, counts = None):
        """Method that generates region data.

        Args:
            vals (list, optional): list containing different values the column could be. Defaults to None.
            counts (list, optional): list containing probability of different values the column could be. Defaults to None.

        Returns:
            ndarray: numpy array containing regions of each customer.
        """        
        if vals == None:
            vals = ["London", "Manchester", "Birmingham", "Edinburgh", "Bristol", "Leeds", "N/A", "", np.nan]
        if counts == None:
            counts = [667, 286, 209, 286, 231, 231, 30, 20, 10]
        total_data = np.repeat(vals, counts)
        self.rng.shuffle(total_data)
        return total_data 
    
    def num_support_tickets(self, lam = None, range = None):
        """_summary_

        Args:
            lam (int, optional): lambda for poisson distribution. Defaults to None.
            range (int, optional): range for the date. Defaults to None.

        Returns:
            ndarray: numpy array containing number of support tickets for each customer.
        """        
        if lam == None:
            lam = 1.2
        if range == None:
            range = [0,10]
        tickets = self.rng.poisson(lam=lam, size=self.n)
        tickets = np.clip(tickets, range[0], range[1])

        return np.round(tickets).astype(int) 
    
    def days_since_last_order(self, last_order_date, ref = "2024-09-01"):
        """Method generating days since last order

        Args:
            last_order_date (int): last order date to calculate elapsed days.
            ref (str, optional): date for calculating elapsed. Defaults to "2024-09-01".

        Returns:
            ndarray: numpy array containing days since last order for each customer.
        """        
        reference_date = pd.Timestamp(ref)
        days_elapsed = (reference_date - pd.Series(last_order_date)).dt.days
        return days_elapsed 
    
    def churned(self, days_elapsed):
        """Method generating churned

        Args:
            days_elapsed (int): days elapsed since last order

        Returns:
            ndarray: numpy array containing if a customer churned or not.
        """        
        return (days_elapsed > 60).astype(int)

In [33]:
class messup:
    def __init__(self, n = 1970, seed = 41):
        """constuctor

        Args:
            n (int, optional): number of the dataframe that needs messing up. Defaults to 1970.
            seed (int, optional): seed for NumPy's default_rng. Defaults to 41.
        """        
        self.n = n
        self.rng = np.random.default_rng(seed=seed)

    def messup_age(self, data, missing = None, low = None, high = None):
        """Method for messing up age data

        Args:
            data (dataframe): dataset
            missing (int, optional): number of nan to insert. Defaults to None.
            low (int, optional): low. Defaults to None.
            high (int, optional): high. Defaults to None.

        Returns:
            dataframe : pandas dataframe of age column messed up
        """        
        if missing == None:
            missing = int(np.ceil(self.n*0.05))
        if low == None:
            low = 5
        if high == None:
            high = 3

        nan_list = [np.nan]*missing
        data_to_add = np.concatenate([nan_list, [5, 8, 11, 3, 1, 135, 142, 121]])
        total_add_count = len(data_to_add)
        replacement_indices = self.rng.choice(self.n, size=total_add_count, replace = False)
        data.iloc[replacement_indices, data.columns.get_loc('age')] = data_to_add 
        return data 
    
    def messy_email_open_rate(self, data, count = None, max = None):
        """Method for messing up email open rate data

        Args:
            data (dataframe): dataset
            count (int, optional): number of indices to replace. Defaults to None.
            max (int, optional): max value for uniform dist. Defaults to None.

        Returns:
            dataframe : pandas dataframe of email open rate column messed up
        """        
        if max == None:
            max = 1.5 
        if count == None:
            count = 25 
        
        indices_to_replace = self.rng.choice(self.n, size=count, replace=False)

        messy_prob_rates = self.rng.uniform(1.05, max, size=count)

        data.iloc[indices_to_replace, data.columns.get_loc('email_open_rate')] = messy_prob_rates

        return data 
    
    def messy_avg_order_value(self, data, ghost_entry = None, entry_error_base = None, num_flaw = 5):
        """Method for messing up average order value data

        Args:
            data (dataframe): dataset
            ghost_entry (int, optional): payment processing ghost entries. Defaults to None.
            entry_error_base (_type_, optional): test orders or data entry errors. Defaults to None.
            num_flaw (int, optional): number to insert per ghost entry and entry error base. Defaults to 5.

        Returns:
            dataframe : pandas dataframeof average order value column messed up
        """        
        if ghost_entry == None:
            ghost_entry = 0.01
        if entry_error_base == None:
            entry_error_base = 901

        total_flaws = num_flaw*2 
        indices_to_replace = self.rng.choice(self.n, size = total_flaws, replace = False)

        ghost_entry_vals = [ghost_entry]*num_flaw 
        entry_error_vals = self.rng.uniform(entry_error_base, 25000, size=num_flaw)
        total_flaw = np.concatenate([ghost_entry_vals, entry_error_vals])
        self.rng.shuffle(total_flaw)

        # add to our program
        data.iloc[indices_to_replace, data.columns.get_loc('avg_order_value')] = total_flaw

        return data

    def messy_total_orders(self, data, num_flaw = 20):
        """Method for messing up total orders data

        Args:
            data (dataframe): dataset
            num_flaw (int, optional): number of 0s to be added despite last order date being occupied. Defaults to 20.

        Returns:
            dataframe : pandas dataframe of total orders column messed up
        """        
        indices_to_replace = self.rng.choice(self.n, size = num_flaw, replace = False)
        zeroes = np.zeros(num_flaw)

        #replace
        data.iloc[indices_to_replace, data.columns.get_loc('total_orders')] = zeroes

        return data 
    
    def messy_last_order_date(self, data, num = 160, rate = False):
        """Method for messing up last order date data

        Args:
            data (dataframe): dataset
            num (int, optional): number of nans to insert indicating customers who signed up but never placed a second order, or whose order history failed to sync. Defaults to 160.
            rate (bool, optional): if percentage is given, calculate num from that. Defaults to False.

        Returns:
            dataframe : pandas dataframe of last order date column messed up
        """        
        if rate:
            num = int(self.n * rate) 
        
        valid_indices = np.where(data['total_orders'].values > 0)[0]
        indices_to_replace = self.rng.choice(valid_indices, size = num, replace = False)

        data.iloc[indices_to_replace, data.columns.get_loc('last_order_date')] = np.nan 
        return data

    def messy_signup_date(self, data, future_start_date = None, rate = None):
        """Method for messing up sign up date data

        Args:
            data (dataframe): dataset
            future_start_date (str, optional): simulate a CRM sync error , start date of future dates to add to data. Defaults to None.
            rate (int, optional): if perentage if given calculate num to replace otherwise its 15. Defaults to None.

        Returns:
            dataframe : pandas dataframe of sign up date column messed up
        """        
        if future_start_date == None:
            future_start_date = "2025-01-01" 
        if rate == None:
            num_to_replace = 15 
        else:
            num_to_replace = int(self.n * rate)
        
        indices_to_replace = self.rng.choice(self.n, size=num_to_replace, replace=False)
        
        # random date generation
        start_date  =  np.datetime64(future_start_date)
        total_days = num_to_replace*10 

        random_offsets = self.rng.integers(0, total_days, num_to_replace)

        random_messy_dates = start_date + np.timedelta64(1, "D") * random_offsets

        data.iloc[indices_to_replace, data.columns.get_loc('signup_date')] = random_messy_dates 

        return data 
    

    def insert_duplicates(self, data, num_of_duplicates = 30):
        """Method for inserting duplicates data

        Args:
            data (dataframe): dataset
            num_of_duplicates (int, optional): number of duplicate rows to add to dataset. Defaults to 30.

        Returns:
            dataframe : pandas dataframe after inserting duplicates
        """        

        # 1. sample duplicate rows
        duplicates = data.sample(n = num_of_duplicates, replace=True)
        
        # 2. combine
        messy_data = pd.concat([data, duplicates], ignore_index=True) 

        # 3. Sort by stable id to keep duplicates together and only sort by id 
        messy_data = messy_data.sort_values(
            by = "customer_id",
            kind = "stable",
        ).reset_index(drop = True)

        return messy_data

In [34]:
import numpy as np
import pandas as pd

# 1. Initialize the generator for 1970 rows
gen = generate(n=1970, seed=42)

# 2. Build the initial clean DataFrame
# Note: We generate 'dates' twice to fill both 'signup_date' and 'last_order_date'
signup_dates = gen.dates("2023-01-01", "2024-01-01")
last_order_dates = gen.dates("2024-01-02", "2024-08-30")
days_elapsed = gen.days_since_last_order(last_order_dates, ref="2024-09-01")

data_dict = {
    "customer_id": gen.customer_id(),
    "signup_date": signup_dates,
    "last_order_date": last_order_dates,
    "total_orders": gen.total_orders(),
    "avg_order_value": gen.avg_order_value(mean=50, std=15, clip=[5, 500]),
    "email_open_rate": gen.email_open_rate(alpha=2, beta=5),
    "customer_segment": gen.customer_segment(),
    "subscription_status": gen.subscription_status(),
    "age": gen.age(),
    "region": gen.region(),
    "num_support_tickets": gen.num_support_tickets(),
    "days_since_last_order": days_elapsed,
    "churned": gen.churned(days_elapsed)
}

df = pd.DataFrame(data_dict)
print(f"Initial clean dataframe shape: {df.shape}")

# 3. Initialize the messup class (matching n=1970)
modifier = messup(n=1970, seed=41)

# 4. Sequentially apply the messiness constraints
df = modifier.messup_age(df)
df = modifier.messy_email_open_rate(df)
df = modifier.messy_avg_order_value(df)
df = modifier.messy_total_orders(df)
df = modifier.messy_last_order_date(df)
df = modifier.messy_signup_date(df)

print(f"Dataframe shape after messiness: {df.shape}")

# 5. Finally, insert the 30 duplicates
df_final = modifier.insert_duplicates(df, num_of_duplicates=30)

print(f"Final dataframe shape (with duplicates): {df_final.shape}")

# 6. Quick sanity check verification
print("\n--- Verification Checks ---")
print(f"Total Rows (Expected 2000): {len(df_final)}")
print(f"Total Duplicate Rows Found: {df_final.duplicated(keep=False).sum()}")
print(f"Null values in 'age': {df_final['age'].isnull().sum()}")

Initial clean dataframe shape: (1970, 13)
Dataframe shape after messiness: (1970, 13)
Final dataframe shape (with duplicates): (2000, 13)

--- Verification Checks ---
Total Rows (Expected 2000): 2000
Total Duplicate Rows Found: 60
Null values in 'age': 100


In [35]:
# This counts how many identical rows are in each duplicate group
print(df_final.groupby(list(df_final.columns)).size().value_counts())
print(df_final.groupby(list(df_final.columns), dropna=False).size().value_counts())

1    1692
2      28
Name: count, dtype: int64
1    1940
2      30
Name: count, dtype: int64


In [36]:
df_final.head()

,customer_id,signup_date,last_order_date,total_orders,avg_order_value,email_open_rate,customer_segment,subscription_status,age,region,num_support_tickets,days_since_last_order,churned
0,CUST-0001,2025-01-03,2024-07-20,5,90.53,0.281393,NEW,Active,35.0,Leeds,0,43,0
1,CUST-0002,2023-10-11,2024-08-11,5,43.01,0.511968,loyal,Cancelled,21.0,London,1,21,0
2,CUST-0003,2023-08-28,2024-06-15,5,43.76,0.212491,Loyal,active,43.0,Manchester,0,78,1
3,CUST-0004,2023-06-10,2024-03-24,5,58.83,0.134714,new,Active,34.0,Leeds,1,161,1
4,CUST-0005,2023-06-08,2024-05-16,3,39.77,0.076015,New,Active,39.0,London,3,108,1


In [40]:
df_final.describe()

,signup_date,last_order_date,total_orders,avg_order_value,email_open_rate,age,num_support_tickets,days_since_last_order,churned
count,2000,1839,2000.000000,2000.000000,2000.000000,1900.000000,2000.000000,2000.000000,2000.000000
mean,2023-07-07 05:51:21,2024-05-02 05:08:30,3.917500,85.369781,0.294396,34.717895,1.222500,121.711500,0.754000
min,2023-01-01 00:00:00,2024-01-02 00:00:00,0.000000,0.010000,0.007543,1.000000,0.000000,2.000000,0.000000
25%,2023-04-01 00:00:00,2024-02-29 12:00:00,3.000000,39.522500,0.159952,28.000000,0.000000,61.000000,1.000000
50%,2023-07-04 00:00:00,2024-05-01 00:00:00,4.000000,49.575000,0.264872,34.000000,1.000000,123.000000,1.000000
75%,2023-10-07 00:00:00,2024-07-02 00:00:00,5.000000,60.052500,0.386599,39.000000,2.000000,184.000000,1.000000
max,2025-05-30 00:00:00,2024-08-30 00:00:00,12.000000,19358.689042,1.444883,142.000000,6.000000,243.000000,1.000000
std,NaN,NaN,1.778279,734.469120,0.193941,9.630968,1.119205,69.624148,0.430786


In [42]:
df_final.to_csv("../data/raw/customer_churn_raw.csv", index=False)

In [19]:
# check if last order date is always equals or above signup date
temp = pd.DataFrame({'signup': signup_date, 'last_order': last_order_date})

is_valid = (temp['last_order'] >= temp['signup']).all()

if is_valid:
    print("✅ Validation Passed: No customer ordered before signing up!")
else:
    print("❌ Validation Failed: There are timeline anomalies in the data.")

✅ Validation Passed: No customer ordered before signing up!
